# Housing Price Prediction Model

This notebook loads the preprocessed data and trains models to predict housing prices.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plot style
sns.set_style("whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Load Processed Data

We load the `training_processed.csv` and `test_processed.csv` files created by the preprocessing script.

In [ ]:
# Load the processed data
try:
    train_df = pd.read_csv("../data/processed_data/training_processed.csv")
    test_df = pd.read_csv("../data/processed_data/test_processed.csv")
    
    print(f"Training data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("\nPlease make sure you have run the 'preprocess_data.py' script successfully.")

Training data shape: (222723, 26)
Test data shape: (55681, 26)


## 2. Separate Features (X) and Target (y)

Create `X_train`, `y_train`, `X_test`, and `y_test`.

-   The **target (y)** is the `price` column.
-   The **features (X)** are all columns *except* for `price` and any of the `_original` columns that are added for reference.

In [7]:
# Define the target variable
target_col = 'price'

# Identify feature columns
# We want all columns that are NOT the target and NOT the original unscaled columns
feature_cols = [col for col in train_df.columns if col != target_col and not col.endswith('_original')]

# Separate features and target
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Number of features being used: {len(feature_cols)}")
print(f"Target variable: {target_col}")
print(f"\nFirst 5 features:\n{feature_cols[:5]}")

Number of features being used: 16
Target variable: price

First 5 features:
['bed', 'bath', 'acre_lot', 'house_size', 'latitude']


## 3. Model 1: Linear Regression (Baseline)

We started with a simple Linear Regression model to establish a baseline performance.

In [10]:
print("--- Training Baseline: Linear Regression ---")

# Initialize and train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluate the model
r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f"Linear Regression R²: {r2_lr:.4f}")
print(f"Linear Regression RMSE: ${rmse_lr:,.2f}")
print(f"Linear Regression MAE: ${mae_lr:,.2f}")

--- Training Baseline: Linear Regression ---


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. Model 2: Random Forest Regressor

Now let's try a more powerful, tree-based model. A Random Forest is excellent for tabular data and can capture complex, non-linear relationships.

In [ ]:
print("\n--- Training Advanced Model: Random Forest Regressor ---")

# Initialize and train the model
# n_jobs=-1 uses all available CPU cores for faster training
# We set hyperparameters like max_depth and min_samples_leaf to prevent overfitting
rf_model = RandomForestRegressor(n_estimators=100, 
                                 random_state=42, 
                                 n_jobs=-1, 
                                 max_depth=20, 
                                 min_samples_leaf=5)

print("Fitting Random Forest... (This may take a moment)")
rf_model.fit(X_train, y_train)
print("Model fitting complete.")

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"\nRandom Forest R²: {r2_rf:.4f}")
print(f"Random Forest RMSE: ${rmse_rf:,.2f}")
print(f"Random Forest MAE: ${mae_rf:,.2f}")

## 5. Visual Evaluation

A scatter plot of Actual vs. Predicted values is a great way to see how well the models are performing. A perfect model would have all dots on the red dashed line.

In [ ]:
print("\n--- Visualizing Model Performance ---")

plt.figure(figsize=(14, 6))

# Plot Random Forest predictions
plt.subplot(1, 2, 1)
# Use alpha=0.3 to see density
sns.scatterplot(x=y_test, y=y_pred_rf, alpha=0.3, s=10) 
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red', linewidth=2)
plt.title(f"Random Forest Predictions (R²: {r2_rf:.4f})")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.ticklabel_format(style='plain', axis='both') # Disable scientific notation

# Plot Linear Regression predictions
plt.subplot(1, 2, 2)
sns.scatterplot(x=y_test, y=y_pred_lr, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red', linewidth=2)
plt.title(f"Linear Regression Predictions (R²: {r2_lr:.4f})")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.ticklabel_format(style='plain', axis='both')

plt.tight_layout()
plt.show()

## 6. Feature Importance

Let's see which features the Random Forest model considered most important.

In [ ]:
print("\n--- Random Forest Feature Importances ---")

# Get feature importances
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# Display top 20 features
print("Top 20 most important features:")
print(feature_importance_df.head(20))

# Plot top 20 features
plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=feature_importance_df.head(20), palette='viridis')
plt.title("Top 20 Feature Importances (Random Forest)")
plt.show()